Primera celda: recuperar nuestros fragmentos
Como las variables de un notebook no se comparten automáticamente con otro, vamos a guardar los fragmentos desde el primer notebook para poder reutilizarlos.

In [3]:
import json
from langchain_core.documents import Document

# Abrimos el archivo generado en el notebook 01
with open("fragmentos.json", "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

# Reconstruimos los fragmentos como objetos Document
fragmentos = []

for item in datos:
    documento = Document(
        page_content=item["texto"],
        metadata=item["metadata"]
    )
    fragmentos.append(documento)

# Verificamos la cantidad de fragmentos recuperados
print(f"Fragmentos recuperados: {len(fragmentos)}")

# Mostramos de dónde proviene el primer fragmento
print("Fuente:", fragmentos[0].metadata)

Fragmentos recuperados: 778
Fuente: {'fuente': 'ATAJO_Dip Allanamientos-digital-a 150.pdf', 'pagina': 1}


Paso 12. Conectar el modelo de embeddings

In [4]:
from langchain_ollama import OllamaEmbeddings

# Creamos el modelo de embeddings
# Utilizamos bge-m3, que ya descargamos con Ollama
modelo_embeddings = OllamaEmbeddings(
    model="bge-m3:latest"
)

# Probamos el modelo con una pregunta de nuestro proyecto
pregunta = "¿Qué derechos tiene un paciente para acceder a su historia clínica?"

# Convertimos la pregunta en un vector numérico
vector = modelo_embeddings.embed_query(pregunta)

# Verificamos el resultado
print("Modelo de embeddings funcionando correctamente")
print("Dimensiones del vector:", len(vector))
print("Primeros 5 valores:", vector[:5])

Modelo de embeddings funcionando correctamente
Dimensiones del vector: 1024
Primeros 5 valores: [-0.05318057, -0.033337522, -0.019514145, -0.0039790855, 0.0026618855]


Paso 13. Crear la base vectorial ChromaDB

In [5]:
from langchain_chroma import Chroma

# Creamos la base vectorial persistente
base_vectorial = Chroma(
    collection_name="cannabis_derechos_bge_m3",
    embedding_function=modelo_embeddings,
    persist_directory="./chroma_db"
)

# Mostramos cuántos fragmentos contiene inicialmente
print("Base vectorial creada correctamente.")
print("Fragmentos almacenados:", base_vectorial._collection.count())

Base vectorial creada correctamente.
Fragmentos almacenados: 0


Paso 14. Cargar los fragmentos en ChromaDB

In [6]:
# Verificamos que la colección esté vacía
cantidad_actual = base_vectorial._collection.count()

if cantidad_actual == 0:

    # Cargamos los fragmentos en grupos de 50
    # para evitar procesarlos todos al mismo tiempo
    tamano_lote = 50

    for inicio in range(0, len(fragmentos), tamano_lote):

        lote = fragmentos[inicio:inicio + tamano_lote]

        base_vectorial.add_documents(lote)

        print(
            f"Fragmentos cargados: "
            f"{min(inicio + tamano_lote, len(fragmentos))}"
            f"/{len(fragmentos)}"
        )

else:
    print(f"La colección ya contiene {cantidad_actual} fragmentos.")
    print("No se agregaron documentos para evitar duplicados.")

# Comprobamos el total almacenado
print("Total en ChromaDB:", base_vectorial._collection.count())

Fragmentos cargados: 50/778
Fragmentos cargados: 100/778
Fragmentos cargados: 150/778
Fragmentos cargados: 200/778
Fragmentos cargados: 250/778
Fragmentos cargados: 300/778
Fragmentos cargados: 350/778
Fragmentos cargados: 400/778
Fragmentos cargados: 450/778
Fragmentos cargados: 500/778
Fragmentos cargados: 550/778
Fragmentos cargados: 600/778
Fragmentos cargados: 650/778
Fragmentos cargados: 700/778
Fragmentos cargados: 750/778
Fragmentos cargados: 778/778
Total en ChromaDB: 778


Paso 15. Crear nuestra función buscar()

In [7]:
# Función para buscar información en nuestra base vectorial

def buscar(pregunta, k=3):

    resultados = base_vectorial.similarity_search(
        pregunta,
        k=k
    )

    for numero, documento in enumerate(resultados, start=1):

        print(f"\n--- RESULTADO {numero} ---")

        print("Fuente:", documento.metadata["fuente"])
        print("Página:", documento.metadata["pagina"])

        print("\nTexto recuperado:")
        print(documento.page_content[:700])

    return resultados

In [8]:
pregunta = "¿Qué derechos tiene un paciente para acceder a su historia clínica?"

resultados = buscar(pregunta)


--- RESULTADO 1 ---
Fuente: ley_26529_derechos_paciente.pdf
Página: 3

Texto recuperado:
los responsables que tendrán a su cargo la guarda de la misma.
ARTICULO 14. — Titularidad. El paciente es el titular de la historia clínica. A su simple
requerimiento debe suministrársele copia de la misma, autenticada por autoridad competente de
la institución asistencial. La entrega se realizará dentro de las cuarenta y ocho (48) horas de
solicitada, salvo caso de emergencia.
ARTICULO 15. — Asientos. Sin perjuicio de lo establecido en los artículos precedentes y de lo
que disponga la reglamentación, en la historia clínica se deberá asentar:
a) La fecha de inicio de su confección;
8/11/26, 8:21 INFOLEG
https://servicios.infoleg.gob.ar/infolegInternet/anexos/160000-164999/160432/norma.htm 3/

--- RESULTADO 2 ---
Fuente: ley_26529_derechos_paciente.pdf
Página: 1

Texto recuperado:
principalmente las relacionadas con sus condiciones socioculturales, de género, de pudor y a su
intimidad, cualquiera s